In [3]:
# ============================================================
# PUD-Berechnung + Polygon-Raster (1km x 1km)
# Kilimanjaro Landscape Bilder
# ============================================================
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, box

# ============================================================
# CELL 1: Daten laden + filtern
# ============================================================
CLASSIFICATION_CSV = r"C:\Users\Lukas\masterthesis\thesis_2026\data\Classification_Individual_CoEco_Landscape\final_class_individual_coeco_landscape_LINEARPROBE.csv"
METADATA_CSV       = r"C:\Users\Lukas\masterthesis\thesis_2026\data\final_flickr_dataset_with_metadata.csv"
OUTPUT_DIR         = r"C:\Users\Lukas\masterthesis\thesis_2026\data\PUD_Analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Nur Landscape-Bilder
df_class     = pd.read_csv(CLASSIFICATION_CSV)
df_landscape = df_class[df_class["pred_label"] == "landscape"].copy()
print(f"Landscape-Bilder: {len(df_landscape)}")

# Photo_ID extrahieren
df_landscape["Photo_ID"] = (
    df_landscape["file_name"]
    .str.extract(r"image_(\d+)\.jpg")
    .astype(int)
)

# Join mit Metadaten
df_meta = pd.read_csv(METADATA_CSV)
df = df_landscape.merge(
    df_meta[["Photo_ID", "owner", "datetaken", "latitude", "longitude"]],
    on="Photo_ID", how="left"
)
print(f"Nach Join: {len(df)} Bilder | Ohne Metadaten: {df['owner'].isna().sum()}")


# ============================================================
# CELL 2: Zeitfilter + Datum parsen
# ============================================================
df["date"]      = pd.to_datetime(df["datetaken"], errors="coerce")
df["year"]      = df["date"].dt.year
df              = df[df["year"] >= 2004].copy()
df["date_only"] = df["date"].dt.date
print(f"Nach Zeitfilter (≥2004): {len(df)} Bilder")
print(f"Zeitraum: {df['year'].min()} – {df['year'].max()}")


# ============================================================
# CELL 3: In UTM Zone 37S projizieren + Rasterzelle zuweisen
# ============================================================
df = df.dropna(subset=["latitude", "longitude"]).copy()

geometry = [Point(lon, lat) for lon, lat in zip(df["longitude"], df["latitude"])]
gdf      = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
gdf      = gdf.to_crs("EPSG:32737")

CELL_SIZE = 1000  # 1km

gdf["cell_x"] = (gdf.geometry.x // CELL_SIZE) * CELL_SIZE
gdf["cell_y"] = (gdf.geometry.y // CELL_SIZE) * CELL_SIZE
gdf["cell_id"] = (
    gdf["cell_x"].astype(int).astype(str) + "_" +
    gdf["cell_y"].astype(int).astype(str)
)
print(f"Einzigartige Rasterzellen: {gdf['cell_id'].nunique()}")


# ============================================================
# CELL 4: PUD-Berechnung
# ============================================================
gdf["user_day"] = gdf["owner"].astype(str) + "_" + gdf["date_only"].astype(str)

puds = gdf.groupby(["cell_id","cell_x","cell_y"]).agg(
    PUD_total      = ("user_day",  "nunique"),
    n_images       = ("Photo_ID",  "count"),
    n_unique_users = ("owner",     "nunique"),
    lat_mean       = ("latitude",  "mean"),
    lon_mean       = ("longitude", "mean"),
    year_min       = ("year",      "min"),
    year_max       = ("year",      "max"),
).reset_index()

n_years = df["year"].max() - 2004 + 1
puds["PUD_annual_mean"]   = puds["PUD_total"] / n_years
puds["PUD_per_user_year"] = puds["PUD_annual_mean"] / puds["n_unique_users"]

print(f"\n=== PUD-ERGEBNISSE ===")
print(f"Rasterzellen mit mind. 1 PUD: {len(puds)}")
print(f"Gesamte PUDs:                 {puds['PUD_total'].sum()}")
print(f"Max. PUDs in einer Zelle:     {puds['PUD_total'].max()}")
print(f"Ø jährliche PUDs pro Zelle:   {puds['PUD_annual_mean'].mean():.3f}")
print(f"Analysezeitraum:              2004–{df['year'].max()} ({n_years} Jahre)")
print(f"\nDeskriptive Statistik:")
print(puds[["PUD_total","PUD_annual_mean","PUD_per_user_year",
            "n_images","n_unique_users"]].describe().round(3))


# ============================================================
# CELL 5: Polygon-Raster erzeugen (1km x 1km Quadrate)
# ============================================================
polygons = []
for _, row in puds.iterrows():
    x0 = row["cell_x"]
    y0 = row["cell_y"]
    polygons.append(box(x0, y0, x0 + CELL_SIZE, y0 + CELL_SIZE))

puds_poly      = gpd.GeoDataFrame(puds, geometry=polygons, crs="EPSG:32737")
puds_poly_wgs84 = puds_poly.to_crs("EPSG:4326")

print(f"\nPolygon-Raster erstellt: {len(puds_poly)} Zellen à 1km×1km")


# ============================================================
# CELL 6: Speichern
# ============================================================
# CSV
pud_csv = os.path.join(OUTPUT_DIR, "landscape_puds_1km_grid_polygons.csv")
puds.to_csv(pud_csv, index=False)
print(f"✅ CSV gespeichert: {pud_csv}")

# GeoPackage UTM (für räumliche Analysen)
gpkg_utm = os.path.join(OUTPUT_DIR, "landscape_puds_1km_grid_polygons_utm.gpkg")
puds_poly.to_file(gpkg_utm, driver="GPKG")
print(f"✅ GeoPackage (UTM 37S) gespeichert: {gpkg_utm}")

# GeoPackage WGS84 (für QGIS / direkte Visualisierung)
gpkg_wgs = os.path.join(OUTPUT_DIR, "landscape_puds_1km_grid_polygons_wgs84.gpkg")
puds_poly_wgs84.to_file(gpkg_wgs, driver="GPKG")
print(f"✅ GeoPackage (WGS84) gespeichert: {gpkg_wgs}")

print(f"\nTop 10 Zellen nach PUD_total:")
print(puds.nlargest(10, "PUD_total")[[
    "cell_id","PUD_total","PUD_annual_mean",
    "n_images","n_unique_users","lat_mean","lon_mean"
]].to_string(index=False))

Landscape-Bilder: 3422
Nach Join: 3422 Bilder | Ohne Metadaten: 0
Nach Zeitfilter (≥2004): 3349 Bilder
Zeitraum: 2004 – 2024
Einzigartige Rasterzellen: 228

=== PUD-ERGEBNISSE ===
Rasterzellen mit mind. 1 PUD: 228
Gesamte PUDs:                 1055
Max. PUDs in einer Zelle:     61
Ø jährliche PUDs pro Zelle:   0.220
Analysezeitraum:              2004–2024 (21 Jahre)

Deskriptive Statistik:
       PUD_total  PUD_annual_mean  PUD_per_user_year  n_images  n_unique_users
count    228.000          228.000            228.000   228.000         228.000
mean       4.627            0.220              0.054    14.689           3.912
std        8.187            0.390              0.024    37.116           6.515
min        1.000            0.048              0.048     1.000           1.000
25%        1.000            0.048              0.048     1.000           1.000
50%        1.000            0.048              0.048     3.000           1.000
75%        4.000            0.190              0.048  